# VoiceDiary AI — Live Cloud GPU Platform
### Bilingual Classroom Lecture Note-Taking & Speaker Diarization Engine
**VoiceDiary © 2026 Abdul Sarim Khan. All Rights Reserved.**

> **Quick Start:** Runtime → Run all (`Ctrl+F9`) · Record live or upload audio — real-time transcription with GPU acceleration!

In [ ]:
!pip install -q --no-cache-dir faster-whisper speechbrain gradio soundfile torchaudio


In [ ]:

import os, time, tempfile, json, re, urllib.request, gc
import numpy as np
import soundfile as sf
import gradio as gr
import torch
import torchaudio
from faster_whisper import WhisperModel
from speechbrain.inference.speaker import EncoderClassifier

# ─── Hardware Acceleration ───
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (AVX2)"
device_type = "cuda" if torch.cuda.is_available() else "cpu"
compute_dtype = "float16" if device_type == "cuda" else "int8"
print(f"⚡ Hardware: {gpu_name} | Compute: {compute_dtype}")

# ─── Model Hub Cache ───
_model_cache = {}
def get_model(name):
    if name not in _model_cache:
        print(f"Loading Whisper model: {name}…")
        os.makedirs("/content/models/whisper", exist_ok=True)
        _model_cache[name] = WhisperModel(name, device=device_type, compute_type=compute_dtype,
            num_workers=2, download_root="/content/models/whisper")
    return _model_cache[name]

print("Pre-warming Large-v3-Turbo default model…")
get_model("large-v3-turbo")

print("Pre-warming SpeechBrain ECAPA-TDNN neural diarizer…")
os.makedirs("/content/models/ecapa", exist_ok=True)
embedder = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="/content/models/ecapa", run_opts={"device": device_type})
print("✓ VoiceDiary AI Engine initialized.")

# ─── Roman Urdu Dictionary ───
_roman = {
    'آپ':'aap','کیسے':'kaisay','ہیں':'hain','کیا':'kya','کر':'kar','رہے':'rahay',
    'ہو':'ho','میں':'main','ہوں':'hoon','یہ':'yeh','وہ':'woh','نہیں':'nahi',
    'ٹھیک':'theek','شکریہ':'shukriya','سلام':'salam','بہت':'bohot','اچھا':'acha',
    'سمجھ':'samajh','سبق':'sabaq','سوال':'sawaal','جواب':'jawab','استاد':'ustaad'
}
def to_roman(t):
    return " ".join(_roman.get(re.sub(r'[\u064B-\u065F\u0670]','',w),w) for w in t.split())

def load_16k_from_file(path):
    try:
        wav, sr = torchaudio.load(path)
        if wav.shape[0] > 1: wav = wav.mean(0, keepdim=True)
        if sr != 16000: wav = torchaudio.transforms.Resample(sr, 16000)(wav)
        return wav.squeeze().numpy().astype(np.float32)
    except Exception:
        d, sr = sf.read(path)
        if d.ndim > 1: d = d.mean(1)
        d = d.astype(np.float32)
        if sr != 16000:
            n = int(len(d)*16000/sr)
            d = np.interp(np.linspace(0,len(d),n,endpoint=False),np.arange(len(d)),d).astype(np.float32)
        return d

COLORS = ['#6366F1','#10B981','#F59E0B','#EC4899','#06B6D4','#8B5CF6','#F97316','#38BDF8']
MODEL_MAP = {
    'Large-v3-Turbo (809M)': 'large-v3-turbo',
    'Whisper Large-v3 (1.5B)': 'large-v3',
    'Whisper Base (74M)': 'base',
    'Whisper Tiny (39M)': 'tiny',
    'Whisper Small (244M)': 'small',
    'Whisper Medium (769M)': 'medium',
}
LANG_MAP = {
    'Bilingual (Urdu + English)': (None, False),
    'Pure Urdu Script (اردو)': ('ur', False),
    'English Only': ('en', False),
    'Roman Urdu (Latin)': ('ur', True),
}

# ─── Core Transcription & Diarization Logic ───
def process_audio_array(data, model_choice, lang_choice, spk1_name, spk2_name, spk3_name, thresh_pct, vad_ms):
    if data is None or len(data) < 4000:
        empty = """<div class='vd-empty'>
          <svg width='54' height='54' viewBox='0 0 24 24' fill='none' stroke='currentColor' stroke-width='1.5' opacity='.35'>
            <path d='M21 15a2 2 0 0 1-2 2H7l-4 4V5a2 2 0 0 1 2-2h14a2 2 0 0 1 2 2z'/>
          </svg>
          <p>Lecture transcript will stream here</p>
          <span>Speak into the microphone or upload an audio recording</span>
        </div>"""
        sidebar = """<div class='vd-empty-sm'>
          <svg width='36' height='36' viewBox='0 0 24 24' fill='none' stroke='currentColor' stroke-width='1.5' opacity='.35'>
            <path d='M17 21v-2a4 4 0 0 0-4-4H5a4 4 0 0 0-4 4v2'/><circle cx='9' cy='7' r='4'/>
            <path d='M23 21v-2a4 4 0 0 0-3-3.87'/><path d='M16 3.13a4 4 0 0 1 0 7.75'/>
          </svg>
          <p>No active speaker profiles</p>
          <span>Start recording to build neural voiceprints</span>
        </div>"""
        export_empty = """<div class='vd-export-row'>
          <span class='vd-dl disabled'>Markdown (.md)</span>
          <span class='vd-dl disabled'>Plain Text (.txt)</span>
          <span class='vd-dl disabled'>Subtitles (.srt)</span>
          <span class='vd-dl disabled'>JSON (.json)</span>
        </div>"""
        return empty, sidebar, export_empty, "", None, None, None

    t0 = time.time()
    dur = len(data) / 16000.0
    mkey = MODEL_MAP.get(model_choice, 'large-v3-turbo')
    model = get_model(mkey)
    target_lang, is_roman = LANG_MAP.get(lang_choice, (None, False))
    thresh = float(thresh_pct) / 100.0

    # Custom Speaker Names Mapping
    name_map = {
        1: (spk1_name or "").strip() or "Professor (Speaker 1)",
        2: (spk2_name or "").strip() or "Student (Speaker 2)",
        3: (spk3_name or "").strip() or "Speaker 3"
    }

    try:
        segs, info = model.transcribe(data, beam_size=1, best_of=1, temperature=0.0,
            language=target_lang, without_timestamps=False, vad_filter=True,
            vad_parameters=dict(min_silence_duration_ms=int(vad_ms)))
    except Exception as e:
        return f"<div class='vd-empty'><p>Transcription error: {e}</p></div>", "", "", "", None, None, None

    profiles = {}; nxt = 1
    html_parts = []; plain = []; srt_parts = []; json_arr = []; si = 1

    for seg in segs:
        txt = seg.text.strip()
        if not txt: continue
        if is_roman: txt = to_roman(txt)
        s0, s1 = seg.start, seg.end
        chunk = data[int(s0*16000):int(s1*16000)]
        spk = 1
        if len(chunk) >= 8000:
            try:
                with torch.inference_mode():
                    w = torch.from_numpy(chunk).float().unsqueeze(0).to(device_type)
                    e = embedder.encode_batch(w).squeeze().detach().cpu().numpy()
                    en = e / (np.linalg.norm(e) or 1.)
                bid, bsim = None, -1.
                for sid, embs in profiles.items():
                    ms_ = max(float(np.dot(en, x)) for x in embs) if embs else 0
                    if ms_ > bsim: bsim, bid = ms_, sid
                if bid and bsim >= thresh:
                    spk = bid
                    if len(profiles[spk]) < 50: profiles[spk].append(en)
                else:
                    spk = nxt; profiles[spk] = [en]; nxt += 1
            except: pass

        display_name = name_map.get(spk, f"Speaker {spk}")
        c = COLORS[(spk-1) % len(COLORS)]
        ts = f"{int(s0//60):02d}:{int(s0%60):02d}"
        urdu = any('\u0600' <= ch <= '\u06FF' for ch in txt)
        txt_class = "vd-txt vd-rtl" if urdu else "vd-txt"

        html_parts.append(f"""<div class="vd-seg" style="border-left-color:{c};">
  <div class="vd-seg-body">
    <div class="vd-seg-meta">
      <span class="vd-dot" style="background:{c};box-shadow:0 0 8px {c};"></span>
      <span class="vd-spk-lbl" style="color:{c};">{display_name}</span>
      <span class="vd-time">[{ts}]</span>
    </div>
    <div class="{txt_class}">{txt}</div>
  </div>
</div>""")
        plain.append(f"[{ts}] {display_name}: {txt}")
        def srt_ts(s):
            return f"{int(s//3600):02d}:{int(s%3600//60):02d}:{int(s%60):02d},{int((s-int(s))*1000):03d}"
        srt_parts.append(f"{si}\n{srt_ts(s0)} --> {srt_ts(s1)}\n[{display_name}]: {txt}\n")
        json_arr.append({"speaker": display_name, "id": spk, "start": round(s0,2), "end": round(s1,2), "time": ts, "text": txt})
        si += 1

    elapsed = time.time() - t0

    # Speaker sidebar rendering
    sb = []
    for sid, embs in profiles.items():
        cc = COLORS[(sid-1) % len(COLORS)]
        sname = name_map.get(sid, f"Speaker {sid}")
        sb.append(f"""<div class="vd-spk-card">
  <div class="vd-spk-av" style="background:{cc};box-shadow:0 0 14px {cc}55">S{sid}</div>
  <div class="vd-spk-info">
    <div class="vd-spk-name">{sname}</div>
    <div class="vd-spk-meta">{len(embs)} centroid voiceprint{'s' if len(embs)!=1 else ''}</div>
  </div>
</div>""")
    if not sb:
        sb = ["<div class='vd-empty-sm'><p>No active speaker profiles</p></div>"]

    stats = f"""<div class="vd-stats">
  <span>⚡ {mkey} · {gpu_name} ({compute_dtype.upper()})</span>
  <span>{dur:.1f}s audio processed in {elapsed:.1f}s ({dur/max(.01,elapsed):.1f}× real-time)</span>
</div>"""

    full_transcript = "\n".join(html_parts) + stats if html_parts else """<div class='vd-empty'><p>No speech detected</p><span>Try speaking closer to the microphone</span></div>"""

    # Generate Export Files
    files = {}
    for ext, content in [
        ('.md',   f"# VoiceDiary Lecture Notes\n**Model:** {mkey} | **Compute:** {gpu_name}\n\n" + "\n\n".join(plain)),
        ('.txt',  "\n".join(plain)),
        ('.srt',  "\n".join(srt_parts)),
        ('.json', json.dumps(json_arr, indent=2, ensure_ascii=False))
    ]:
        tmp = tempfile.NamedTemporaryFile(mode='w', suffix=ext, delete=False, encoding='utf-8', prefix='VoiceDiary_')
        tmp.write(content); tmp.close()
        files[ext] = tmp.name

    export_html = f"""<div class="vd-export-row">
  <a class="vd-dl" href="/file={files['.md']}" download>
    <svg width="16" height="16" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><path d="M14 2H6a2 2 0 0 0-2 2v16a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V8z"/><polyline points="14 2 14 8 20 8"/><line x1="12" y1="18" x2="12" y2="12"/><line x1="9" y1="15" x2="15" y2="15"/></svg>
    Markdown (.md)
  </a>
  <a class="vd-dl" href="/file={files['.txt']}" download>
    <svg width="16" height="16" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><path d="M14 2H6a2 2 0 0 0-2 2v16a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V8z"/><polyline points="14 2 14 8 20 8"/></svg>
    Plain Text (.txt)
  </a>
  <a class="vd-dl" href="/file={files['.srt']}" download>
    <svg width="16" height="16" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><rect x="2" y="2" width="20" height="20" rx="2"/><path d="M8 10h8M8 14h5"/></svg>
    Subtitles (.srt)
  </a>
  <a class="vd-dl" href="/file={files['.json']}" download>
    <svg width="16" height="16" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><polyline points="16 18 22 12 16 6"/><polyline points="8 6 2 12 8 18"/></svg>
    JSON Data (.json)
  </a>
</div>"""

    return full_transcript, "\n".join(sb), export_html, "\n".join(plain), files.get('.md'), files.get('.txt'), files.get('.srt')

# ─── Gemini 2.5 Flash Summarizer ───
def gemini_summary(text, key):
    if not text or not text.strip():
        return "⚠️ *Please transcribe a lecture first before generating an AI summary.*"
    api_key = (key or "").strip() or os.environ.get("GEMINI_API_KEY","")
    if not api_key:
        return "⚠️ *Please enter your Gemini API Key in the left sidebar to generate structured study notes.*"
    
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={api_key}"
    prompt = (
        f"You are VoiceDiary AI, an elite university academic note-taker. "
        f"Analyze this multi-speaker classroom lecture transcript (Urdu and English code-switching) "
        f"and generate a comprehensive study guide formatted in clean Markdown:\n\n"
        f"## 📋 Executive Lecture Overview\n"
        f"(High-level synopsis of core themes)\n\n"
        f"## 🧠 Key Academic Concepts & Definitions\n"
        f"(Bulleted breakdown of technical points and explanations)\n\n"
        f"## 🎯 Critical Points for Exams\n"
        f"(Important concepts highlighted by the instructor)\n\n"
        f"## 💡 Notable Classroom Q&A & Discussion\n"
        f"(Summary of student questions and teacher answers)\n\n"
        f"Transcript:\n{text}"
    )
    payload = {"contents":[{"parts":[{"text": prompt}]}]}
    try:
        req = urllib.request.Request(url, data=json.dumps(payload).encode(),
                                     headers={"Content-Type":"application/json"})
        with urllib.request.urlopen(req, timeout=35) as r:
            res = json.loads(r.read())
            return res["candidates"][0]["content"]["parts"][0]["text"]
    except Exception as e:
        return f"❌ *Gemini AI Error: {e}*"

# ─── High-Density Modern CSS ───
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800&family=Fira+Code:wght@400;500;600&family=Noto+Nastaliq+Urdu:wght@400;700&display=swap');

/* ── VIEWPORT RESET ── */
html, body { margin: 0 !important; padding: 0 !important; height: 100% !important; background: #080C14 !important; }

.gradio-container {
  max-width: 100% !important;
  width: 100% !important;
  min-height: 100vh !important;
  margin: 0 !important;
  padding: 0 24px 32px 24px !important;
  background: #080C14 !important;
  font-family: 'Plus Jakarta Sans', -apple-system, BlinkMacSystemFont, sans-serif !important;
  color: #F8FAFC !important;
}

/* Background Ambient Glow */
.gradio-container::before {
  content: '';
  position: fixed; inset: 0; z-index: 0; pointer-events: none;
  background:
    radial-gradient(ellipse 80% 45% at 50% -10%, rgba(99,102,241,.18) 0%, transparent 65%),
    radial-gradient(ellipse 60% 35% at 85% 90%, rgba(139,92,246,.10) 0%, transparent 55%);
}

/* Chrome Cleanup */
.gradio-container .block,
.gradio-container .form,
.gradio-container .gap { box-shadow: none !important; border: none !important; background: transparent !important; }
.gr-group, .gr-box, .gr-panel { background: transparent !important; border: none !important; box-shadow: none !important; }
footer, .footer, .gr-footer { display: none !important; }

/* ── INPUTS & CONTROLS (Larger & Clearer) ── */
.gradio-container select,
.gradio-container input[type=text],
.gradio-container input[type=password],
.gradio-container textarea {
  background: rgba(15,23,42,.75) !important;
  border: 1px solid rgba(255,255,255,.12) !important;
  border-radius: 10px !important;
  color: #F8FAFC !important;
  font-family: inherit !important;
  font-size: 14px !important;
  font-weight: 500 !important;
  padding: 11px 14px !important;
  transition: all 0.2s ease !important;
}
.gradio-container select:focus,
.gradio-container input:focus {
  border-color: #6366F1 !important;
  box-shadow: 0 0 12px rgba(99,102,241,.35) !important;
  outline: none !important;
}

/* Labels */
.gradio-container label > span,
.gradio-container .label-wrap span {
  color: #94A3B8 !important;
  font-size: 12px !important;
  font-weight: 700 !important;
  letter-spacing: .06em !important;
  text-transform: uppercase !important;
}

/* Sliders */
input[type=range] { accent-color: #6366F1 !important; }
.gradio-container .gr-slider input[type=number] {
  background: rgba(15,23,42,.9) !important;
  border: 1px solid rgba(255,255,255,.15) !important;
  color: #F8FAFC !important;
  border-radius: 8px !important;
  font-weight: 600 !important;
  font-size: 13px !important;
}

/* Tabs */
.tab-nav { background: transparent !important; border-bottom: 1px solid rgba(255,255,255,.10) !important; margin-bottom: 16px !important; }
.tab-nav button {
  color: #94A3B8 !important; font-weight: 700 !important; font-size: 14px !important;
  padding: 12px 24px !important; background: transparent !important;
  border: none !important; border-bottom: 2px solid transparent !important;
  border-radius: 0 !important; transition: all .15s !important;
}
.tab-nav button.selected { color: #FFFFFF !important; border-bottom-color: #6366F1 !important; background: rgba(99,102,241,.08) !important; }
.tabitem { background: transparent !important; border: none !important; padding: 4px 0 !important; }

/* Audio Component Fix (Prevents cut-off buttons) */
.gradio-container .audio,
.gradio-container .gr-audio {
  background: rgba(15,23,42,.65) !important;
  border: 1px solid rgba(255,255,255,.10) !important;
  border-radius: 14px !important;
  overflow: visible !important;
  min-height: 85px !important;
  padding: 12px !important;
}

/* Primary Action Buttons */
.gradio-container button.primary,
.vd-btn-primary {
  background: linear-gradient(135deg,#6366F1 0%,#8B5CF6 100%) !important;
  color: #FFFFFF !important; border: none !important;
  font-weight: 700 !important; font-size: 15px !important;
  border-radius: 12px !important; padding: 14px 28px !important;
  box-shadow: 0 0 24px rgba(99,102,241,.35) !important;
  transition: all .2s !important; width: 100% !important; cursor: pointer !important;
}
.gradio-container button.primary:hover,
.vd-btn-primary:hover {
  transform: translateY(-2px) !important;
  box-shadow: 0 0 32px rgba(99,102,241,.55) !important;
}

/* ── HEADER NAVBAR (Edge-to-Edge) ── */
.vd-hdr {
  display: flex; align-items: center; justify-content: space-between;
  padding: 18px 0; margin-bottom: 24px;
  border-bottom: 1px solid rgba(255,255,255,.08);
}
.vd-hdr-left { display: flex; align-items: center; gap: 14px; }
.vd-logo-box {
  width: 44px; height: 44px; border-radius: 12px; flex-shrink: 0;
  background: linear-gradient(135deg,#6366F1,#8B5CF6);
  display: flex; align-items: center; justify-content: center;
  box-shadow: 0 0 20px rgba(99,102,241,.45);
}
.vd-hdr-title { font-size: 22px; font-weight: 800; color: #FFFFFF; letter-spacing: -.02em; }
.vd-hdr-sub { font-size: 13px; color: #94A3B8; font-weight: 500; margin-top: 2px; }
.vd-hw-pill {
  display: inline-flex; align-items: center; gap: 8px;
  padding: 7px 18px; border-radius: 9999px;
  background: rgba(16,185,129,.12); border: 1px solid rgba(16,185,129,.28);
  font-size: 12px; font-weight: 700; color: #10B981;
  font-family: 'Fira Code', monospace;
}
.vd-hw-dot { width: 8px; height: 8px; border-radius: 50%; background: #10B981; box-shadow: 0 0 10px rgba(16,185,129,.8); }

/* ── SIDEBAR SECTION ── */
.vd-sec-lbl {
  font-size: 12px; font-weight: 800; color: #94A3B8;
  letter-spacing: .08em; text-transform: uppercase;
  display: flex; align-items: center; justify-content: space-between;
  margin-bottom: 12px; margin-top: 18px;
}
.vd-sec-lbl:first-child { margin-top: 0; }
.vd-badge-live {
  width: 8px; height: 8px; border-radius: 50%; background: #10B981;
  box-shadow: 0 0 10px rgba(16,185,129,.9); display: inline-block; margin-right: 6px;
  animation: pulse 2s infinite;
}
@keyframes pulse { 0%,100%{opacity:1} 50%{opacity:.4} }

/* Speaker Cards */
.vd-spk-card {
  display: flex; align-items: center; gap: 14px;
  padding: 12px 16px; border-radius: 12px;
  background: rgba(15,23,42,.6); border: 1px solid rgba(255,255,255,.08);
  margin-bottom: 10px; transition: .15s ease;
}
.vd-spk-card:hover { background: rgba(30,41,59,.75); border-color: rgba(255,255,255,.16); transform: translateX(3px); }
.vd-spk-av {
  width: 40px; height: 40px; border-radius: 50%; flex-shrink: 0;
  display: flex; align-items: center; justify-content: center;
  font-weight: 800; font-size: 14px; color: #FFFFFF;
}
.vd-spk-info { flex: 1; min-width: 0; }
.vd-spk-name { font-size: 14px; font-weight: 700; color: #F1F5F9; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }
.vd-spk-meta { font-size: 12px; color: #64748B; margin-top: 2px; }

/* ── TRANSCRIPT SEGMENT CARDS ── */
.vd-transcript-vp {
  background: rgba(10,15,28,.85); border: 1px solid rgba(255,255,255,.08);
  border-radius: 16px; padding: 22px; min-height: 380px; max-height: 520px;
  overflow-y: auto; backdrop-filter: blur(16px);
}
.vd-seg {
  margin-bottom: 14px; border-radius: 12px;
  background: rgba(15,23,42,.75); border: 1px solid rgba(255,255,255,.08);
  border-left-width: 5px; border-left-style: solid;
  overflow: hidden; transition: .15s ease;
}
.vd-seg:hover { background: rgba(30,41,59,.85); border-color: rgba(255,255,255,.16); }
.vd-seg-body { padding: 16px 20px; }
.vd-seg-meta { display: flex; align-items: center; gap: 10px; margin-bottom: 8px; }
.vd-dot { width: 8px; height: 8px; border-radius: 50%; display: inline-block; }
.vd-spk-lbl { font-size: 14px; font-weight: 700; }
.vd-time { font-size: 12px; color: #64748B; font-family: 'Fira Code', monospace; }
.vd-txt { font-size: 16px; line-height: 1.7; color: #F1F5F9; word-break: break-word; font-weight: 400; }
.vd-rtl { direction: rtl; text-align: right; font-family: 'Noto Nastaliq Urdu', serif; font-size: 20px; line-height: 2.2; color: #FFFFFF; }

/* Stats Bar */
.vd-stats {
  margin-top: 16px; padding-top: 14px; border-top: 1px solid rgba(255,255,255,.08);
  display: flex; justify-content: space-between; align-items: center;
  font-size: 12px; color: #64748B; font-family: 'Fira Code', monospace;
}

/* Empty States */
.vd-empty {
  display: flex; flex-direction: column; align-items: center; justify-content: center;
  padding: 80px 24px; text-align: center; color: #475569; gap: 12px;
}
.vd-empty p { font-size: 17px; font-weight: 700; color: #94A3B8; margin: 0; }
.vd-empty span { font-size: 13px; color: #64748B; }
.vd-empty-sm {
  display: flex; flex-direction: column; align-items: center;
  padding: 28px 12px; text-align: center; color: #475569; gap: 8px;
}
.vd-empty-sm p { font-size: 13px; font-weight: 600; color: #94A3B8; margin: 0; }
.vd-empty-sm span { font-size: 11px; color: #64748B; }

/* ── EXPORT BUTTONS ── */
.vd-export-row { display: flex; gap: 12px; flex-wrap: wrap; margin-top: 6px; }
.vd-dl {
  display: inline-flex; align-items: center; gap: 8px;
  padding: 11px 18px; border-radius: 10px;
  background: rgba(255,255,255,.05); border: 1px solid rgba(255,255,255,.12);
  color: #E2E8F0; font-size: 14px; font-weight: 600;
  text-decoration: none; transition: .15s ease; cursor: pointer;
}
.vd-dl:hover { background: rgba(99,102,241,.18); border-color: rgba(99,102,241,.45); color: #A5B4FC; transform: translateY(-1px); }
.vd-dl.disabled { opacity: 0.4; cursor: not-allowed; pointer-events: none; }

/* Dividers */
.vd-divider { border: none; border-top: 1px solid rgba(255,255,255,.08); margin: 18px 0; }

/* AI Summary Card */
.vd-summary-card {
  background: rgba(15,23,42,.75); border: 1px solid rgba(255,255,255,.10);
  border-radius: 16px; padding: 24px; font-size: 15px; line-height: 1.7;
  color: #F1F5F9; backdrop-filter: blur(16px);
}
"""

with gr.Blocks(title="VoiceDiary — AI Bilingual Lecture & Diarization Engine", css=CSS,
               theme=gr.themes.Default(primary_hue="indigo", neutral_hue="slate")) as demo:

    # Hidden states
    transcript_state = gr.State("")
    stream_buffer_state = gr.State({"audio": np.array([], dtype=np.float32), "last_processed_len": 0})

    # ── 1. HEADER (Full-Width Clean Navbar) ──
    gr.HTML(f"""
    <div class="vd-hdr">
      <div class="vd-hdr-left">
        <div class="vd-logo-box">
          <svg width="22" height="22" viewBox="0 0 24 24" fill="none" stroke="white" stroke-width="2.2">
            <path d="M12 1a3 3 0 0 0-3 3v8a3 3 0 0 0 6 0V4a3 3 0 0 0-3-3z"/>
            <path d="M19 10v2a7 7 0 0 1-14 0v-2"/>
            <line x1="12" y1="19" x2="12" y2="23"/><line x1="8" y1="23" x2="16" y2="23"/>
          </svg>
        </div>
        <div>
          <div class="vd-hdr-title">VoiceDiary</div>
          <div class="vd-hdr-sub">Bilingual Classroom Lecture Note-Taking &amp; Neural Diarization Engine</div>
        </div>
      </div>
      <div class="vd-hw-pill">
        <span class="vd-hw-dot"></span>
        {gpu_name} &nbsp;·&nbsp; Tensor Cores {compute_dtype.upper()}
      </div>
    </div>""")

    # ── 2. TWO-COLUMN HIGH-EFFICIENCY LAYOUT ──
    with gr.Row(equal_height=False):

        # ── LEFT SIDEBAR (Controls & Profiles) ──
        with gr.Column(scale=3, min_width=280):
            # Speaker Profiles & Dynamic Naming
            gr.HTML("<div class='vd-sec-lbl'><span>Speakers &amp; Neural Profiles</span><span><span class='vd-badge-live'></span>LIVE</span></div>")
            sidebar_out = gr.HTML(value="""<div class='vd-empty-sm'>
              <svg width='36' height='36' viewBox='0 0 24 24' fill='none' stroke='currentColor' stroke-width='1.5' opacity='.35'>
                <path d='M17 21v-2a4 4 0 0 0-4-4H5a4 4 0 0 0-4 4v2'/><circle cx='9' cy='7' r='4'/>
                <path d='M23 21v-2a4 4 0 0 0-3-3.87'/><path d='M16 3.13a4 4 0 0 1 0 7.75'/>
              </svg>
              <p>No active speaker profiles</p>
              <span>Start recording to build neural voiceprints</span>
            </div>""")

            # Speaker Rename Inputs
            with gr.Accordion("✏️ Rename Speaker Labels", open=False):
                spk1_name = gr.Textbox(value="Professor", label="Speaker 1 Display Name", placeholder="e.g. Professor Mamuna")
                spk2_name = gr.Textbox(value="Student 1", label="Speaker 2 Display Name", placeholder="e.g. Sarim Khan")
                spk3_name = gr.Textbox(value="Student 2", label="Speaker 3 Display Name", placeholder="e.g. Class Representative")

            gr.HTML("<hr class='vd-divider'>")
            gr.HTML("<div class='vd-sec-lbl'><span>AI Engine &amp; Whisper Model</span></div>")
            model_dd = gr.Dropdown(choices=list(MODEL_MAP.keys()),
                value='Large-v3-Turbo (809M)', label='Active Whisper Model')
            lang_dd = gr.Dropdown(choices=list(LANG_MAP.keys()),
                value='Bilingual (Urdu + English)', label='Language Output Mode')

            gr.HTML("<hr class='vd-divider'>")
            gr.HTML("<div class='vd-sec-lbl'><span>Diarization Sensitivity</span></div>")
            thresh_sl = gr.Slider(20, 70, 32, step=1, label='Cosine Similarity Threshold (%)')
            vad_sl = gr.Slider(150, 600, 280, step=10, label='VAD Silence Gap (ms)')

            gr.HTML("<hr class='vd-divider'>")
            gr.HTML("<div class='vd-sec-lbl'><span>Google Gemini AI (BYOK)</span></div>")
            gemini_key = gr.Textbox(placeholder='Paste Gemini API Key (AIzaSy…)', type='password',
                                    label='Gemini API Key', container=False)

        # ── RIGHT MAIN WORKSPACE ──
        with gr.Column(scale=9, min_width=520):
            # Audio Input Tabs
            with gr.Tabs():
                with gr.TabItem("🎙️ Live Classroom Microphone"):
                    audio_mic = gr.Audio(sources=["microphone"], type="filepath",
                                         label="Click microphone to start recording lecture speech",
                                         show_label=False)
                with gr.TabItem("📁 Upload Pre-Recorded Lecture"):
                    audio_file = gr.Audio(sources=["upload"], type="filepath",
                                          label="Upload classroom audio (.wav, .mp3, .m4a, .flac)",
                                          show_label=False)

            transcribe_btn = gr.Button("⚡ Transcribe & Diarize Lecture (GPU Accelerated)",
                                        variant="primary", elem_classes=["vd-btn-primary"])

            gr.HTML("<div class='vd-sec-lbl' style='margin-top:24px;'><span>Classroom Lecture Transcript</span></div>")
            transcript_out = gr.HTML(
                value="""<div class='vd-empty'>
                  <svg width='54' height='54' viewBox='0 0 24 24' fill='none' stroke='currentColor' stroke-width='1.5' opacity='.35'>
                    <path d='M21 15a2 2 0 0 1-2 2H7l-4 4V5a2 2 0 0 1 2-2h14a2 2 0 0 1 2 2z'/>
                  </svg>
                  <p>Lecture transcript will stream here</p>
                  <span>Speak into the microphone or upload an audio recording</span>
                </div>""",
                elem_classes=["vd-transcript-vp"])

            gr.HTML("<div class='vd-sec-lbl' style='margin-top:20px;'><span>Export Lecture Notes</span></div>")
            export_html_out = gr.HTML(value="""<div class='vd-export-row'>
              <span class='vd-dl disabled'>Markdown (.md)</span>
              <span class='vd-dl disabled'>Plain Text (.txt)</span>
              <span class='vd-dl disabled'>Subtitles (.srt)</span>
              <span class='vd-dl disabled'>JSON (.json)</span>
            </div>""")

            # Hidden file references for Gradio download serving
            with gr.Row(visible=False):
                f_md  = gr.File()
                f_txt = gr.File()
                f_srt = gr.File()

            gr.HTML("<hr class='vd-divider' style='margin:26px 0 20px 0;'>")
            gr.HTML("<div class='vd-sec-lbl'><span>AI Study Summary &amp; Flashcards (Gemini 2.5 Flash)</span></div>")
            ai_btn = gr.Button("✨ Generate AI Lecture Summary & Study Guide",
                               variant="primary", elem_classes=["vd-btn-primary"])
            ai_out = gr.Markdown(value="*AI study summary and key concepts will be generated here after clicking above.*",
                                 elem_classes=["vd-summary-card"])

    # ── 3. EVENT BINDINGS ──
    all_inputs  = [audio_mic, audio_file, model_dd, lang_dd, spk1_name, spk2_name, spk3_name, thresh_sl, vad_sl]
    all_outputs = [transcript_out, sidebar_out, export_html_out, transcript_state, f_md, f_txt, f_srt]

    def run_mic_finish(mic, _f, mod, lang, s1, s2, s3, th, vad):
        if not mic: return process_audio_array(None, mod, lang, s1, s2, s3, th, vad)
        data = load_16k_from_file(mic)
        return process_audio_array(data, mod, lang, s1, s2, s3, th, vad)

    def run_file_btn(mic, fpath, mod, lang, s1, s2, s3, th, vad):
        target = fpath if fpath else mic
        if not target: return process_audio_array(None, mod, lang, s1, s2, s3, th, vad)
        data = load_16k_from_file(target)
        return process_audio_array(data, mod, lang, s1, s2, s3, th, vad)

    # Auto-run immediately when user stops recording
    audio_mic.stop_recording(fn=run_mic_finish, inputs=all_inputs, outputs=all_outputs)
    # Manual run button (for file upload or re-running with new speaker names)
    transcribe_btn.click(fn=run_file_btn, inputs=all_inputs, outputs=all_outputs)
    # Gemini AI Summary
    ai_btn.click(fn=gemini_summary, inputs=[transcript_state, gemini_key], outputs=[ai_out])

demo.queue(max_size=20).launch(share=True, inline=True, debug=False, show_error=True)
